# Exploratory Data Analysis

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

plt.style.use('seaborn-v0_8-darkgrid')

## Load Dataset
Load from CSV (or generate if not found using classifier.generate_training_data)

In [ ]:
try:
    df = pd.read_csv('../data/processed/cipher_dataset.csv')
    print('Dataset loaded successfully.')
except FileNotFoundError:
    print('Dataset not found.')
    df = pd.DataFrame()

## Class Distribution
Bar chart of cipher type counts

In [ ]:
plt.figure(figsize=(10, 6))
sns.countplot(data=df, x='label', hue='label', palette='viridis', legend=False)
plt.title('Cipher Type Distribution')
plt.xlabel('Cipher Type')
plt.ylabel('Count')
plt.show()

## Feature Distributions
Histograms of IC, entropy, chi-squared by cipher type (overlaid)
Box plots of key features grouped by cipher type

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.histplot(data=df, x='ioc', hue='label', element='step', ax=axes[0])
axes[0].set_title('Index of Coincidence Distribution')

sns.histplot(data=df, x='entropy', hue='label', element='step', ax=axes[1])
axes[1].set_title('Entropy Distribution')

sns.histplot(data=df, x='chi_square', hue='label', element='step', ax=axes[2])
axes[2].set_title('Chi-Square Distribution')

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.boxplot(data=df, x='label', y='ioc', hue='label', ax=axes[0])
axes[0].set_title('Index of Coincidence by Cipher')
axes[0].tick_params(axis='x', rotation=45)

sns.boxplot(data=df, x='label', y='entropy', hue='label', ax=axes[1])
axes[1].set_title('Entropy by Cipher')
axes[1].tick_params(axis='x', rotation=45)

sns.boxplot(data=df, x='label', y='chi_square', hue='label', ax=axes[2])
axes[2].set_title('Chi-Square by Cipher')
axes[2].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## Correlation Analysis
Correlation heatmap of top features
Feature importance from a quick Random Forest

In [ ]:
plt.figure(figsize=(12, 10))
numeric_df = df.drop(columns=['label'])
sns.heatmap(numeric_df.corr(), cmap='coolwarm', center=0)
plt.title('Feature Correlation Heatmap')
plt.show()

In [ ]:
X = df.drop(columns=['label'])
y = df['label']

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X, y)

importances = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)

plt.figure(figsize=(10, 6))
importances.head(15).plot(kind='bar')
plt.title('Top 15 Feature Importances (Random Forest)')
plt.ylabel('Importance')
plt.show()

## Dimensionality Reduction
PCA 2D scatter plot colored by cipher type
t-SNE 2D scatter plot colored by cipher type

In [ ]:
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X)
df_pca = pd.DataFrame({'PC1': X_pca[:, 0], 'PC2': X_pca[:, 1], 'label': y})

plt.figure(figsize=(10, 8))
sns.scatterplot(data=df_pca, x='PC1', y='PC2', hue='label', palette='tab10')
plt.title('PCA: 2D Projection of Features')
plt.show()

In [ ]:
tsne = TSNE(n_components=2, random_state=42)
X_tsne = tsne.fit_transform(X)
df_tsne = pd.DataFrame({'Dim1': X_tsne[:, 0], 'Dim2': X_tsne[:, 1], 'label': y})

plt.figure(figsize=(10, 8))
sns.scatterplot(data=df_tsne, x='Dim1', y='Dim2', hue='label', palette='tab10')
plt.title('t-SNE: 2D Projection of Features')
plt.show()

## Key Observations
- Distribution plots show clear separation in some features (e.g. IoC for transposition vs substitution).
- PCA and t-SNE indicate that some classes cluster well while others overlap.
- Top features heavily influence the Random Forest.